# 02 · Crosswalk: CPS census codes → O*NET

Part 2 of 5. CPS occupations are 2010 Census codes; O*NET uses O*NET-SOC 2019 (built on the 2018 SOC). The chain is census → 2010 SOC → 2018 SOC → O*NET.

**Reads:** `data/raw/cenocc2010.xlsx` (BLS), `data/raw/soc_2010_to_2018_crosswalk.xlsx` (BLS), `data/raw/cps_00001.csv`, `data/derived/job-features.parquet`
**Writes:** `data/derived/census-onet-crosswalk.parquet`, `data/derived/census-soc2010-crosswalk.parquet`

**Results**
- 517 of 535 census codes reach O*NET; every O*NET occupation is reachable.
- The 18 lost codes are mostly "all other" residual categories, plus legislators and financial analysts (O*NET 31.0 has no ratings for either). 1.6% of employed workers fall in unmapped codes.
- Broad SOC groups (trailing zeros, e.g. `11-2020`) are expanded by prefix match against the detailed 2010 codes, which also handles the 2010→2018 splits and merges.

In [1]:
import numpy as np
import pandas as pd

cps = pd.read_csv("../data/raw/cps_00001.csv")
feat = pd.read_parquet("../data/derived/job-features.parquet").set_index("O*NET-SOC Code")
abilities = pd.read_csv("../data/raw/db_31_0_csv/abilities.csv")
print(cps.shape, feat.shape)

(1083340, 26) (910, 110)


## Census → 2010 SOC (BLS file)

In [2]:
# CROSSWALK

xw = pd.read_excel('../data/raw/cenocc2010.xlsx', header=None)
xw = xw.iloc[:, :3]
xw.columns = ['title', 'census', 'soc']

# keep real occupations only: single 4-digit census codes, drop group/range rows
xw = xw.dropna(subset=['census'])
xw['census'] = xw['census'].astype(str).str.strip()
xw = xw[xw['census'].str.fullmatch(r'\d{4}', na=False)]

# one row per (census, soc) in case a cell lists several SOC codes
xw['soc'] = xw['soc'].str.split(r'[,;]\s*')
xw = xw.explode('soc')
xw['soc'] = xw['soc'].str.strip()
xw['census'] = xw['census'].astype(int)

print(xw.shape, xw['census'].nunique(), 'census codes')

# 1) SOC values that aren't plain detailed codes (broad groups, X placeholders, etc.)
odd = xw[~xw['soc'].str.fullmatch(r'\d{2}-\d{3}[1-9]')]
print(len(odd), 'non-detailed SOC entries')
print(odd.head(30))

# 2) CPS codes with no match in the crosswalk (IPUMS harmonization leaks)
cps_codes = set(cps['OCC2010'].unique()) | set(cps['OCC10LY'].unique())
missing = sorted(cps_codes - set(xw['census']) - {9999})
print(len(missing), 'CPS codes not in crosswalk:', missing)

(551, 3) 535 census codes
116 non-detailed SOC entries
                                                 title  census      soc
17                        Marketing and sales managers      50  11-2020
30                            Education administrators     230  11-9030
49   Claims adjusters, appraisers, examiners, and i...     540  13-1030
52                             Human resources workers     630  13-1070
69                 Credit counselors and loan officers     910  13-2070
83                        Computer support specialists    1050  15-1150
92      Miscellaneous mathematical science occupations    1240  15-2090
95                            Architects, except naval    1300  17-1010
96     Surveyors, cartographers, and photogrammetrists    1310  17-1020
103               Electrical and electronics engineers    1410  17-2070
105  Industrial engineers, including health and safety    1430  17-2110
113                                           Drafters    1540  17-3010
114      

## 2010 SOC → 2018 SOC → O*NET

In [3]:
soc = pd.read_excel('../data/raw/soc_2010_to_2018_crosswalk.xlsx', header=8, dtype=str)
soc = soc.rename(columns={'2010 SOC Code': 'soc2010', '2018 SOC Code': 'soc2018'})[['soc2010', 'soc2018']]
soc = soc.dropna().apply(lambda c: c.str.strip())

# expand census -> detailed 2010 SOC: exact match for detailed codes, prefix match for broad groups
xw['prefix'] = xw['soc'].str.rstrip('0').str.rstrip('-')
detailed_2010 = soc['soc2010'].unique()

rows = []
for _, r in xw.iterrows():
    hits = [c for c in detailed_2010 if c.startswith(r['prefix'])]
    for c in hits:
        rows.append((r['census'], c))
census_soc2010 = pd.DataFrame(rows, columns=['census', 'soc2010']).drop_duplicates()

# 2010 SOC -> 2018 SOC
census_soc2018 = census_soc2010.merge(soc, on='soc2010').drop_duplicates(['census', 'soc2018'])

# 2018 SOC -> O*NET-SOC (O*NET code = 2018 SOC + '.xx')
onet = pd.DataFrame({'onet': feat.index})
onet['soc2018'] = onet['onet'].str[:7]
census_onet = census_soc2018.merge(onet, on='soc2018')[['census', 'onet']].drop_duplicates()

# coverage
print(census_onet['census'].nunique(), 'of', xw['census'].nunique(), 'census codes reach O*NET')
print(len(set(feat.index) - set(census_onet['onet'])), 'O*NET codes unreached')
lost = sorted(set(xw['census']) - set(census_onet['census']))
print(xw[xw['census'].isin(lost)][['census', 'title']].drop_duplicates())

517 of 535 census codes reach O*NET
0 O*NET codes unreached
     census                                              title
15       30                                        Legislators
65      840                                 Financial analysts
122    1660                         Life scientists, all other
150    2060                       Religious workers, all other
191    2960  Media and communication equipment workers, all...
276    4160  Food preparation and serving related workers, ...
306    4650      Personal care and service workers, all other 
328    4965               Sales and related workers, all other
334    5030      Communications equipment operators, all other
342    5165                        Financial clerks, all other
359    5420           Information and record clerks, all other
382    5940  Office and administrative support workers, all...
492    7855                 Food processing workers, all other
513    8220       Metal workers and plastic workers, all o

## What gets lost

In [4]:
# 1) share of employed CPS workers (weighted) whose current occupation is in a lost code
emp = cps[cps['OCC2010'] != 9999]
lost_share = emp.loc[emp['OCC2010'].isin(lost), 'ASECWT'].sum() / emp['ASECWT'].sum()
print(f'{lost_share:.2%} of employed workers are in unmapped census codes')

# 2) trace financial analysts (census 840) through each hop
print(xw[xw['census'] == 840][['census', 'soc', 'prefix']])
print(soc[soc['soc2010'].str.startswith('13-205')])
print([c for c in feat.index if c.startswith('13-205')])
print([c for c in abilities['O*NET-SOC Code'].unique() if c.startswith('13-205')])

1.60% of employed workers are in unmapped census codes
    census      soc   prefix
65     840  13-2051  13-2051
    soc2010  soc2018
64  13-2051  13-2051
65  13-2051  13-2054
66  13-2052  13-2052
67  13-2053  13-2053
['13-2052.00', '13-2053.00']
['13-2052.00', '13-2053.00']


## Save hand-off files

In [5]:
census_onet.to_parquet('../data/derived/census-onet-crosswalk.parquet', index=False)
xw.to_parquet('../data/derived/census-soc2010-crosswalk.parquet', index=False)
print(census_onet.shape, xw.shape, 'saved')

(940, 2) (551, 4) saved
